In [1]:
!rm -rf /kaggle/working/**

In [2]:
!pip install PyYAML

In [3]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import seaborn as sns
import yaml
from collections import defaultdict
from IPython.display import display, HTML

In [4]:
# --- 1. Main Configuration ---

BASELINE_CSV_PATH = '/kaggle/input/r-b-aabb-w-yolov8n-s/yolo_training_results/yolov8s/results.csv'
BASE_INPUT_DIRECTORY = "/kaggle/input/"
OUTPUT_PLOT_DIRECTORY = "/kaggle/working/comparison_plots"

# Map folder names to the desired display names and YAML keys
ARGUMENT_MAPPING = {
    'flipud': 'flipud',
    'fliplr': 'fliplr',
    'translate': 'translate',
    'rotation': 'degrees',        # 'rot' in folder name maps to 'degrees' in YAML
    'c-p': 'copy_paste',     # 'c-p' in folder name maps to 'copy_paste' in YAML
    'mosaic': 'mosaic',
    'scale': 'scale',
    'mixup': 'mixup',
    'pov': 'perspective',
    'shear': 'shear'    
}

In [5]:
# --- 2. Function to Compare Runs and Find the Best ---
def compare_grouped_runs(baseline_csv, aug_group_name, augmented_csvs):
    """
    Compares runs to the baseline, finds the best run, and returns its data including the best value.
    """
    display_name = ARGUMENT_MAPPING.get(aug_group_name, aug_group_name)
    print(f"\n--- Analyzing Group: '{display_name}' vs. Baseline ---")
    
    try:
        df_base = pd.read_csv(baseline_csv)
        df_base.columns = df_base.columns.str.strip()
        base_best_map50 = df_base['metrics/mAP50(B)'].max()

        performance_data = []
        best_run_info = {'score': -1}

        for aug_csv in augmented_csvs:
            args_yaml_path = os.path.join(os.path.dirname(aug_csv), 'args.yaml')
            actual_aug_value = 'N/A'
            if os.path.exists(args_yaml_path):
                with open(args_yaml_path, 'r') as f: args = yaml.safe_load(f)
                param_key = ARGUMENT_MAPPING.get(aug_group_name)
                if param_key and param_key in args:
                    actual_aug_value = args[param_key]

            df_aug = pd.read_csv(aug_csv)
            df_aug.columns = df_aug.columns.str.strip()
            aug_best_map50 = df_aug['metrics/mAP50(B)'].max()
            percent_change = ((aug_best_map50 - base_best_map50) / base_best_map50) * 100

            performance_data.append({
                'Augmentation': display_name, 'Value': actual_aug_value,
                'Best mAP@0.5': aug_best_map50, '% Change vs Baseline': percent_change
            })

            if aug_best_map50 > best_run_info['score']:
                best_run_info = {
                    'score': aug_best_map50, 'df': df_aug, 
                    'name': display_name, 'value': actual_aug_value # --- NEW: Store the best value ---
                }

        comp_df = pd.DataFrame(performance_data)
        display_df = comp_df.copy()
        display_df['% Change vs Baseline'] = display_df['% Change vs Baseline'].map('{:+.2f}%'.format)
        print(display_df.sort_values(by='Best mAP@0.5', ascending=False).to_string(index=False))

        best_run_summary = None
        if best_run_info['score'] > -1:
            best_epoch_df = best_run_info['df'].loc[best_run_info['df']['metrics/mAP50(B)'].idxmax()]
            p, r = best_epoch_df['metrics/precision(B)'], best_epoch_df['metrics/recall(B)']
            f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
            
            best_run_summary = {
                'Augmentation': best_run_info['name'],
                'Best Value': best_run_info['value'], # --- NEW: Add best value to summary ---
                'f1': f1, 'precision': p, 'recall': r,
                'mAP@0.5': best_epoch_df['metrics/mAP50(B)'],
                'mAP@0.5:0.95': best_epoch_df['metrics/mAP50-95(B)']
            }

        return comp_df, best_run_summary
        
    except Exception as e:
        print(f"  [Error] Could not process comparison for group '{display_name}': {e}")
        return pd.DataFrame(), None

In [6]:
# --- 3. Function to Plot Grouped Comparisons ---
def plot_grouped_comparison(baseline_csv, aug_group_name, augmented_csvs, output_dir):
    """
    Generates a single plot comparing the baseline against all values in an augmentation group.
    """
    display_name = ARGUMENT_MAPPING.get(aug_group_name, aug_group_name)
    print(f"🖼️  Generating grouped plot for '{display_name}'...")
    try:
        metrics_to_plot = { 'mAP@0.5': 'metrics/mAP50(B)', 'mAP@0.5:0.95': 'metrics/mAP50-95(B)', 'Precision': 'metrics/precision(B)', 'Recall': 'metrics/recall(B)' }
        fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(12, 18), sharex=True)
        title_name = f"Group Comparison: Baseline vs. {display_name.capitalize()}"
        fig.suptitle(title_name, fontsize=16, weight='bold')

        df_base = pd.read_csv(baseline_csv); df_base.columns = df_base.columns.str.strip()
        for i, (title, column) in enumerate(metrics_to_plot.items()):
            # --- THIS IS THE CORRECTED LINE ---
            axes[i].plot(df_base['epoch'], df_base[column], color='black', linestyle='--', label='Baseline', linewidth=2.5)
            axes[i].set_title(title); axes[i].set_ylabel('Score'); axes[i].grid(True, linestyle=':', alpha=0.6)

        colors = plt.cm.viridis(np.linspace(0, 1, len(augmented_csvs)))
        for idx, aug_csv in enumerate(augmented_csvs):
            args_yaml_path = os.path.join(os.path.dirname(aug_csv), 'args.yaml'); actual_aug_value = 'N/A'
            if os.path.exists(args_yaml_path):
                with open(args_yaml_path, 'r') as f: args = yaml.safe_load(f)
                param_key = ARGUMENT_MAPPING.get(aug_group_name);
                if param_key and param_key in args: actual_aug_value = args[param_key]
            df_aug = pd.read_csv(aug_csv); df_aug.columns = df_aug.columns.str.strip()
            for i, (title, column) in enumerate(metrics_to_plot.items()):
                axes[i].plot(df_aug['epoch'], df_aug[column], color=colors[idx], label=f"Value: {actual_aug_value}", linewidth=2)
        
        for ax in axes: ax.legend()
        axes[-1].set_xlabel('Epoch'); plt.tight_layout(rect=[0, 0.03, 1, 0.96])
        safe_name = re.sub(r'[^a-zA-Z0-9_-]', '', display_name)
        output_path = os.path.join(output_dir, f"GroupComparison_{safe_name}.png"); plt.savefig(output_path); plt.close(fig)
    except Exception as e: print(f"  [Error] Could not create plot for group '{display_name}': {e}")

In [7]:
# --- 4. Function to Create a Summary Heatmap of % Change ---
def create_summary_heatmap(all_results_df, output_dir):
    """
    Creates a heatmap of the '% Change vs Baseline' for all augmentations and values.
    """
    print("\n\n🔥 Generating summary heatmap of Percent Change...")
    if all_results_df.empty: print("  [Skipping Heatmap] No results data to plot."); return
    try:
        all_results_df['Value'] = pd.to_numeric(all_results_df['Value'], errors='coerce'); all_results_df.dropna(subset=['Value'], inplace=True)
        heatmap_data = all_results_df.pivot_table(index='Augmentation', columns='Value', values='% Change vs Baseline')
        plt.figure(figsize=(12, max(6, len(heatmap_data) * 0.8)))
        sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="viridis", linewidths=.5, cbar_kws={'label': '% Change in mAP@0.5 vs. Baseline'})
        plt.title('Augmentation Performance Heatmap (% Change vs. Baseline)', fontsize=16)
        plt.xlabel('Augmentation Value'); plt.ylabel('Augmentation Type'); plt.xticks(rotation=45); plt.yticks(rotation=0)
        output_path = os.path.join(output_dir, "PercentChange_Heatmap.png"); plt.savefig(output_path, bbox_inches='tight'); plt.close()
        print(f"✅ Percent Change heatmap saved to {output_path}")
    except Exception as e: print(f"  [Error] Could not generate the Percent Change heatmap: {e}")

In [8]:
# --- 5. NEW: Function to Create the Final Summary Table Image ---
def create_summary_table_image(summary_data, output_dir):
    """
    Creates a heatmap-styled IMAGE summarizing the best run for each augmentation.
    """
    print("\n\n🏆 Generating final summary table image...")
    if not summary_data: print("  [Skipping Summary Table] No data available."); return
    
    summary_df = pd.DataFrame(summary_data).set_index('Augmentation')
    
    # Separate the numeric metrics from the best value
    best_values = summary_df['Best Value']
    numeric_df = summary_df[['f1', 'precision', 'recall', 'mAP@0.5', 'mAP@0.5:0.95']]
    
    plt.figure(figsize=(10, max(5, len(numeric_df) * 0.6)))
    ax = sns.heatmap(numeric_df, annot=True, fmt=".4f", cmap="Blues", linewidths=.5, cbar=True)
    
    # --- NEW: Create new labels for the y-axis ---
    new_yticklabels = [f"{label.get_text()} (Value: {best_values.get(label.get_text(), 'N/A')})" for label in ax.get_yticklabels()]
    ax.set_yticklabels(new_yticklabels, rotation=0)

    plt.title("Best Augmentation Performance Summary", fontsize=16)
    plt.xlabel("Metrics"); plt.ylabel("Augmentation Type")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout()
    output_path = os.path.join(output_dir, "FinalSummary_Table_Heatmap.png")
    plt.savefig(output_path); plt.close()
    print(f"✅ Final summary table image saved to {output_path}")

In [9]:
# --- 6. Main Execution Logic ---
def main():
    if not os.path.exists(BASELINE_CSV_PATH): print(f"❌ FATAL ERROR: Baseline file not found at '{BASELINE_CSV_PATH}'"); return
        
    os.makedirs(OUTPUT_PLOT_DIRECTORY, exist_ok=True)
    all_results, best_runs_summary = [], []
    
    augmentation_groups = defaultdict(list)
    
    # --- FIX: Robustly find the baseline's parent directory name ---
    try:
        path_parts = BASELINE_CSV_PATH.split('/')
        input_index = path_parts.index('input')
        baseline_dir_name = path_parts[input_index + 1]
        print(f"✅ Baseline model identified. Excluding all runs from directory: '{baseline_dir_name}'")
    except (ValueError, IndexError):
        print(f"❌ FATAL ERROR: Could not parse the baseline directory from the path: {BASELINE_CSV_PATH}")
        return

    for notebook_name in os.listdir(BASE_INPUT_DIRECTORY):
        if not os.path.isdir(os.path.join(BASE_INPUT_DIRECTORY, notebook_name)) or notebook_name == baseline_dir_name:
            continue
        match = re.search(r'(yolo\w+)', notebook_name)
        if not match: continue
        try: aug_type = notebook_name.split(f'{match.group(1)}-')[1]
        except IndexError: continue
        
        search_path = os.path.join(BASE_INPUT_DIRECTORY, notebook_name, 'yolo_training_results', '*', 'results.csv')
        result_files = glob.glob(search_path)
        if result_files: augmentation_groups[aug_type].extend(result_files)
    
    if not augmentation_groups: print("❌ No augmentation runs found to compare."); return

    print(f"✅ Found {len(augmentation_groups)} augmentation group(s) to process.")

    for group_name, csv_paths in augmentation_groups.items():
        plot_grouped_comparison(BASELINE_CSV_PATH, group_name, csv_paths, OUTPUT_PLOT_DIRECTORY)
        results_df, best_run_data = compare_grouped_runs(BASELINE_CSV_PATH, group_name, csv_paths)
        
        if not results_df.empty: all_results.append(results_df)
        if best_run_data: best_runs_summary.append(best_run_data)

    if all_results:
        master_df = pd.concat(all_results, ignore_index=True)
        create_summary_heatmap(master_df, OUTPUT_PLOT_DIRECTORY)
    
    create_summary_table_image(best_runs_summary, OUTPUT_PLOT_DIRECTORY)

    print("\n\n✅ Automation complete.")
    !zip -r comp.zip /kaggle/working/comparison_plots

# Execute the script
main()

✅ Baseline model identified. Excluding all runs from directory: 'r-b-aabb-w-yolov8n-s'
✅ Found 10 augmentation group(s) to process.
🖼️  Generating grouped plot for 'copy_paste'...

--- Analyzing Group: 'copy_paste' vs. Baseline ---
Augmentation  Value  Best mAP@0.5 % Change vs Baseline
  copy_paste    0.1        0.5466               +0.00%
  copy_paste    0.3        0.5466               +0.00%
  copy_paste    0.5        0.5466               +0.00%
🖼️  Generating grouped plot for 'perspective'...

--- Analyzing Group: 'perspective' vs. Baseline ---
Augmentation  Value  Best mAP@0.5 % Change vs Baseline
 perspective 0.0010       0.57863               +5.86%
 perspective 0.0005       0.56032               +2.51%
 perspective 0.0001       0.55702               +1.91%
🖼️  Generating grouped plot for 'degrees'...

--- Analyzing Group: 'degrees' vs. Baseline ---
Augmentation  Value  Best mAP@0.5 % Change vs Baseline
     degrees   10.0       0.55536               +1.60%
     degrees    5.0   

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


✅ Percent Change heatmap saved to /kaggle/working/comparison_plots/PercentChange_Heatmap.png


🏆 Generating final summary table image...
✅ Final summary table image saved to /kaggle/working/comparison_plots/FinalSummary_Table_Heatmap.png


✅ Automation complete.
  adding: kaggle/working/comparison_plots/ (stored 0%)
  adding: kaggle/working/comparison_plots/GroupComparison_flipud.png (deflated 7%)
  adding: kaggle/working/comparison_plots/GroupComparison_copy_paste.png (deflated 9%)
  adding: kaggle/working/comparison_plots/GroupComparison_perspective.png (deflated 7%)
  adding: kaggle/working/comparison_plots/GroupComparison_mosaic.png (deflated 6%)
  adding: kaggle/working/comparison_plots/GroupComparison_degrees.png (deflated 7%)
  adding: kaggle/working/comparison_plots/FinalSummary_Table_Heatmap.png (deflated 6%)
  adding: kaggle/working/comparison_plots/GroupComparison_mixup.png (deflated 6%)
  adding: kaggle/working/comparison_plots/PercentChange_Heatmap.png (deflated 11%)
  add